 # Table of Contents
+ [Import](#Import_0)
+ [Files](#Files_1)
	+ [Input](#Input_2)
	+ [Output](#Output_3)
+ [Download LASER Ontology](#Download_LASER_Ontology_4)
+ [Get LASER tags](#Get_LASER_tags_5)
+ [Save results](#Save_results_6)


<a class="anchor" id="Import_0"></a>
# <span class=title_0 style="color: #4E2C73">Import</span>

In [1]:
import pandas as pd
import sys
import re
import urllib

from pathlib import Path
sys.path.append("../..")

from file_management import check_save_file, get_files_dir

# Resolve project directories
_, INPUT_DIR, OUTPUT_DIR = get_files_dir()


from parse_LASER import get_tags

<a class="anchor" id="Files_1"></a>
# <span class=title_0 style="color: #4E2C73">Files</span>

This section defines:

- Project input/output directories

- Local paths for LASER-related files

- Remote URLs for the LASER database and ontology

Path were the htmls were downloaded: 'https://bitbucket.org/jdwinkler/laser_release/src/master/database_store/'

<a class="anchor" id="Input_2"></a>
## <span class=title_1 style="color: #622870">Input</span>

This section defines:

- Local LASER directory

- Directory containing downloaded LASER HTML files

In [2]:
LASER_FILES = INPUT_DIR / "LASER"
LASER_FILES.mkdir(parents=True, exist_ok=True)

LASER_WEBFILES = INPUT_DIR /"LASER"/"LASER_webfiles"

These HTML files contain references to individual paper records

(e.g. RecordXXXX.txt) for each year

In [3]:
with open(LASER_WEBFILES /"2014.html", "r", encoding='latin') as f:
    laser_2014= f.read()
with open(LASER_WEBFILES /"2015.html", "r", encoding='latin') as f:
    laser_2015= f.read()


Each paper is referenced by a Record*.txt file.

We extract and deduplicate all such references per year.

In [4]:
papers_2014 = re.findall(r"Record.{1,30}\.txt", laser_2014)
papers_2014 = set(papers_2014)
print(len(papers_2014))

papers_2015 = re.findall(r"Record.{1,30}\.txt", laser_2015)
papers_2015 = set(papers_2015)
print(len(papers_2015))


313
143


In [5]:
papers_sets = {"2014":list(papers_2014), "2015":list(papers_2015)}


Define LASER remote resources

- Product ontology: defines LASER product categories

- Database store: contains individual Record*.txt files

In [6]:
url_ontology = (
    "https://bitbucket.org/jdwinkler/laser_release/raw/"
    "f6ce080a8993ee259c4914ce92f83b1f966bab2d/"
    "inputs/Product%20Ontology.txt"
)

database_url = (
    "https://bitbucket.org/jdwinkler/laser_release/raw/"
    "f6ce080a8993ee259c4914ce92f83b1f966bab2d/"
    "database_store/")


<a class="anchor" id="Output_3"></a>
## <span class=title_1 style="color: #622870">Output</span>

This section defines:

- Local output filenames for the LASER ontology

- Extracted LASER tags per paper

In [7]:
OUTPUT_FILE_ONTOLOGY = "LASER_product_ontology.txt"
OUTPUT_FILE_TAGS = "LASER_TAGS.json"
laser_ontology_file = LASER_FILES / OUTPUT_FILE_ONTOLOGY


<a class="anchor" id="Download_LASER_Ontology_4"></a>
# <span class=title_0 style="color: #4E2C73">Download LASER Ontology</span>

In [10]:
if not laser_ontology_file.exists():
    print("Downloading LASER_product_ontology.txt ...")
    urllib.request.urlretrieve(url_ontology, laser_ontology_file)
else:
    print("LASER_product_ontology.txt already exists.")

In [9]:
LASER_ontology = pd.read_csv(laser_ontology_file, sep='\t')

In [10]:
LASER_ontology.head()

,Product,Intended usage,product class
0,(2S)-NARINGENIN,pharmaceuticals,chalcone
1,(2S)-PINOCEMBRIN,food additives,phenolics
2,1_2-PROPANEDIOL,polymers,diols
3,1_3-PROPANEDIOL,polymers,diols
4,1_4-BUTANEDIOL,polymers,diols


<a class="anchor" id="Get_LASER_tags_5"></a>
# <span class=title_0 style="color: #4E2C73">Get LASER tags</span>

This step:

- Iterates over all Record*.txt files listed in papers_sets

- Downloads each record from the LASER database

- Extracts product tags using the get_tags helper function

In [11]:
full_tags = get_tags(papers_sets, database_url)

In [13]:
full_tags = pd.DataFrame.from_dict(full_tags)

<a class="anchor" id="Save_results_6"></a>
# <span class=title_0 style="color: #4E2C73">Save results</span>

In [16]:
_ = check_save_file(
    full_tags,
    OUTPUT_FILE_TAGS,
    "LASER",
    input_dir=True
)

Saved file in: /Users/elisamarquez/Documents/PhD/Utimo/ELISER-StrainDesignDB/files/Input/LASER/LASER_TAGS.json
